#### The aim of this notebook is to build gradually certain particle dynamics regarding its orientation and position.

In [2]:
import sys
from pathlib import Path

import numpy as np 
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Path to src_Jeffery.py
src_path = Path("../Pysimulations").resolve()

if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

import src_Jeffery as jf

#Parameters 
import tomli
from pathlib import Path

parameter_file = Path("parameters.toml")

with open(parameter_file, "rb") as f:
    params = tomli.load(f)

print("parameters.toml loaded correctly")

# PARTICLE PARAMETERS

Length = params["particle"]["length"]
diameter = params["particle"]["diameter"]

swimming_speed = params["particle"]["swimming_speed"]

eta = params["particle"]["eta"]

n_particle = params["particle"]["n_particle"]
n_medium = params["particle"]["n_medium"]


# INITIAL CONDITIONS

R0 = np.array([
    params["initial"]["x0"],
    params["initial"]["y0"],
    params["initial"]["z0"]
], dtype=float)

theta0_deg = params["initial"]["theta0_deg"]
phi0_deg = params["initial"]["phi0_deg"]


# FLOW PARAMETERS

gamma = params["flow"]["gamma"]
plane = params["flow"]["plane"]


# OPTICAL PARAMETERS

wavelength = params["optics"]["wavelength"]
w0 = params["optics"]["w0"]
laser_power = params["optics"]["laser_power"]

beam_axis = np.array(
    params["optics"]["beam_axis"],
    dtype=float
)

e_pol = np.array(
    params["optics"]["polarization"],
    dtype=float
)


# SIMULATION PARAMETERS

t0 = params["simulation"]["t0"]
tf = params["simulation"]["tf"]
n_points = params["simulation"]["n_points"]
rtol = params["simulation"]["rtol"]
atol = params["simulation"]["atol"]

print("PARTICLE")
print("Length =", Length, "m")
print("diameter =", diameter, "m")
print("swimming_speed =", swimming_speed, "m/s")
print("eta =", eta, "Pa s")
print("n_particle =", n_particle)
print("n_medium =", n_medium)

print()

print("INITIAL CONDITIONS")
print("R0 =", R0, "m")
print("theta0 =", theta0_deg, "deg")
print("phi0 =", phi0_deg, "deg")

print()

print("FLOW")
print("gamma =", gamma, "1/s")
print("plane =", plane)

print()

print("OPTICS")
print("wavelength =", wavelength, "m")
print("w0 =", w0, "m")
print("laser_power =", laser_power, "W")
print("beam_axis =", beam_axis)
print("polarization =", e_pol)



print()

print("SIMULATION")
print("t0 =", t0, "s")
print("tf =", tf, "s")
print("n_points =", n_points)

parameters.toml loaded correctly
PARTICLE
Length = 5e-07 m
diameter = 7e-08 m
swimming_speed = 1e-05 m/s
eta = 0.000953 Pa s
n_particle = 1.38
n_medium = 1.33

INITIAL CONDITIONS
R0 = [0. 0. 0.] m
theta0 = 25.0 deg
phi0 = 25.0 deg

FLOW
gamma = 1 1/s
plane = xz

OPTICS
wavelength = 1.31e-06 m
w0 = 1.9e-06 m
laser_power = 0.025 W
beam_axis = [0. 0. 1.]
polarization = [0. 1. 0.]

SIMULATION
t0 = 0.0 s
tf = 300.0 s
n_points = 4000


# Notation

A micro or nano particle will be represented by two main variables:

$$\mathbf{R}(t):\begin{pmatrix}
x\\
y\\
z 
\end{pmatrix}, \qquad\qquad \mathbf{p}(t)
:\begin{pmatrix}
p_x\\
p_y\\
p_z 
\end{pmatrix}$$
where $\mathbf{R}$ is the position of the center of mass, $\mathbf{p}$ is the orientation of the long axis and $||\mathbf{p}||$


## Properties of the particle
The particle is approximated as a prolate spheroid with total length L and diameter D.

The aspect ratio is 

$$c = \frac{L}{D}$$

and the corresponding Bretherton parameter is 

$$\beta = \frac{c^2-1}{c^2+1}$$

The Bretherton parameter becomes 0 for a sphere ($\beta=0$), approaches 1 at the slender limit ($\beta\rightarrow 1$) and approaches -1 at the disk limit ($\beta\rightarrow-1 $).

The volume is calculated through 

$$V = \frac{4}{3}\pi ab^2$$

Using $a= \text{Length}/2$ and $b= \text{diameter}/2$.

The particle's polarizability can be obtained from Clausius-Mossoti:

$$\alpha=3V\varepsilon_0\frac{\varepsilon_r-1}{\varepsilon_r+2} $$

Using

$$\varepsilon_r= \left(\frac{n_p}{n_m} \right)^2$$

Ishimoto, K. (2023). Jeffery’s orbits and microswimmers in flows: a theoretical review. Journal of the Physical Society of Japan, 92(6), 062001.

Levy, O., & Bergman, D. J. (1992). Clausius-Mossotti approximation for a family of nonlinear composites. Physical Review B, 46(11), 7189.

In [3]:
#Particle Geometry
aspect_ratio = Length/diameter


#This is in src_Jeffery
def lambda_ar(c: float) -> float:
    c2 = c * c
    lamb = (c2 - 1.0) / (c2 + 1.0)
    return lamb

beta = jf.lambda_ar(aspect_ratio)
print("Bretherton parameter: ",beta)
print("Aspecto ratio: ",aspect_ratio)

a = Length/2
b = diameter/2

Volume = (4/3)*np.pi*a*(b**2)
print("Volume:", Volume, "m^3")

alpha_scalar = jf.alpha(Volume, n_particle, n_medium)
print("Particle's polarizability:", alpha_scalar)

Bretherton parameter:  0.9615535504119262
Aspecto ratio:  7.142857142857142
Volume: 1.2828170002158323e-21 m^3
Particle's polarizability: 8.483977117647845e-34


The tridimensional orientation vector is described as:
$$\mathbf{p}
:\begin{pmatrix}
\sin\theta\cos\phi\\
\sin\theta\sin\phi\\
\cos\theta 
\end{pmatrix}, \qquad \qquad ||\mathbf{p}|| =  1$$

In [4]:
#Shperical angles to unit vector. 
def sph_to_vec(theta: float, phi: float) -> np.ndarray:
    return np.array([
        np.sin(theta) * np.cos(phi), #d1 = rcos(phi), r = sin(theta)
        np.sin(theta) * np.sin(phi), #d2 = rsin(phi), r = sin(theta)
        np.cos(theta)  #d3, d3´=-theta´sin(theta)
    ], dtype=float)

theta0 = np.deg2rad(theta0_deg)
phi0 = np.deg2rad(phi0_deg)

p0 = jf.sph_to_vec(theta0, phi0)

print("p: ", p0)
print("theta0: ", theta0)
print("phi0: ", phi0)
print("||p0|| =", np.linalg.norm(p0))
    

p:  [0.38302222 0.1786062  0.90630779]
theta0:  0.4363323129985824
phi0:  0.4363323129985824
||p0|| = 1.0


<p align="center">
    <img src="imagenes/Jeffery_simpleshear.png" width="600">
</p>

<p align="center">
    <b>Figure 1.</b> Jeffery orbit in simple shear flow. The red line represent the evolution of the direction through time.
</p>

The shear represented by $\mathbf{u}$, in a simple shear in xz $\left(\mathbf{u}=\dot{\gamma}y,0,0\right)$, Hence
$$\nabla\mathbf{u}=
\begin{pmatrix}
0 & 0 & \dot{\gamma}\\
0 & 0 & 0\\
0 & 0 & 0
\end{pmatrix}$$
Then is decomposed into its symmetric (Deformation/strain,$\mathbf{E}$) and antisymmetric (local rigid-body rotation, $\mathbf{W}$) parts ($\nabla\mathbf{u}=\mathbf{G}$)

$$
\mathbf{E} = \frac{1}{2}
\left(\mathbf{G}+\mathbf{G}^T\right)=
\begin{pmatrix}
0 & 0 & \dot{\gamma}\\
\dot{\gamma} & 0 & 0\\
0 & 0 & 0
\end{pmatrix},
$$

$$
\mathbf{W} = \frac{1}{2}
\left(\mathbf{G}-\mathbf{G}^T\right)=
\begin{pmatrix}
0 & \dot{\gamma} & 0\\
-\dot{\gamma} & 0 & 0\\
0 & 0 & 0
\end{pmatrix}.
$$ 

In [5]:

#Velocity gradient 3x3 matrix G= nabla u, G[i, j] =du_x/dx_j. (x,y,z).
def grad_u_simple_shear(gamma: float, plane: str) -> np.ndarray:
    G = np.zeros((3,3), dtype=float)
    if plane == "xy":
        G[0,1] = gamma
    elif plane == "xz":
        G[0,2] = gamma
    elif plane == "yx":
        G[1, 0] = gamma

    elif plane == "yz":
        G[1, 2] = gamma

    elif plane == "zx":
        G[2, 0] = gamma

    elif plane == "zy":
        G[2, 1] = gamma

    else:
        raise ValueError(
            "plane must be 'xy', 'xz', 'yx', 'yz', 'zx', or 'zy'"
        )

    return G
#----------------------------------------
G = jf.grad_u_simple_shear(gamma,plane)
print("simple shear:")
print(G)

def decompose_grad_u(G):
    E = 0.5 * (G + G.T)
    W = 0.5 * (G - G.T)
    return E, W
E, W = jf.decompose_grad_u(G)

print("symetric part E:")
print(E)

print("antisymetric part W:")
print(W)
G2 = jf.grad_u_extensional(gamma,plane)



print("extensional shear")
print(G2)

simple shear:
[[0. 0. 1.]
 [0. 0. 0.]
 [0. 0. 0.]]
symetric part E:
[[0.  0.  0.5]
 [0.  0.  0. ]
 [0.5 0.  0. ]]
antisymetric part W:
[[ 0.   0.   0.5]
 [ 0.   0.   0. ]
 [-0.5  0.   0. ]]
extensional shear
[[ 1.  0.  0.]
 [ 0.  0.  0.]
 [ 0.  0. -1.]]


#### Direction
The simulations are based on the physics defined by Ishimoto to describe how is the orientationof non-spherical microswimmersin shear through the Jeffrey equation:
$$ \dot{\mathbf{p}}= \mathbf{Wp}+\beta[\mathbf{Ep}-(\mathbf{p}^T\mathbf{Ep})\mathbf{p}] \tag{1}$$

Which describes the angular evolution of a spheroidal body.

In [6]:
def jeffery_rhs_vector(t: float, p: np.ndarray, E: np.ndarray, W: np.ndarray, lam: float) -> np.ndarray:

    p = np.asarray(p, dtype=float)
    Ep = E @ p #Ejk d
    Wp = W @ p #omega * d
    dEp = p @ Ep 
    dp = Wp + lam * (Ep - dEp * p)
    return dp


p_dot_0 = jf.jeffery_rhs_vector(t0,p0,E,W,beta)
print("p0 =", p0)
print("p_dot(0) =", p_dot_0)

p0 = [0.38302222 0.1786062  0.90630779]
p_dot(0) = [ 0.76103669 -0.05961694 -0.30987928]


In [7]:
t_eval = np.linspace(t0,tf,n_points)

sol_jeffery = solve_ivp(
    fun=lambda t, p: jf.jeffery_rhs_vector(t,p,E,W,beta),
    t_span=(t0, tf),
    y0=p0,
    t_eval=t_eval,
    rtol=rtol,
    atol=atol
)
P_raw = sol_jeffery.y.T

norm_raw = np.linalg.norm(P_raw,axis=1)

P = jf.normalize_rows(P_raw)
p0_sim = P[0]
pf = P[-1]

px = P[:, 0]
py = P[:, 1]
pz = P[:, 2]

print("Initial director =", p0_sim)
print("Final director =", pf)

Initial director = [0.38302222 0.1786062  0.90630779]
Final director = [-0.9500432   0.06548931 -0.30517054]


#### Traslational
For the aim of this studies we should consider the center of mass position $\mathbf{R}$. the particle will start at:
$$\mathbf{R}:\begin{pmatrix}
x\\
y\\
z 
\end{pmatrix}$$
and will be modeled as:
$$\dot{\mathbf{R}}=u(\mathbf{R})+v_s\mathbf{p}+\mu_t\mathbf{F}_{opt}(\mathbf{R}) \tag{2}$$
Where $u(\mathbf{R})$ is the contribution of the flow advection, $v_s\mathbf{p}$ is the contribution made by a self-propultion the particle might or might not have and the direction $\mathbf{p}$ will define the direction of swimming and $\mu_t\mathbf{F}_{opt}(\mathbf{R})$ will be the velocity induced by the optical force (given by the "restoring force", $\mathbf{F}_{opt}$) caused by the optical trap which will be trying to get back the particle to the same position. 

#### Potential tweezers

Optical tweezers trapping potential is usually modeled as proportional to the light intensity profile, a gaussian beam; the full potential expression is: 
$$U(r,z)=-U_0\frac{w_0^2}{w(z)^2}exp\left(-\frac{2r^2}{w(z)^2}\right)$$
where:
$$w(z)=w_0\sqrt{1+\left(\frac{z}{z_R}\right)^2}\tag{beam waist}$$
$$z_R=\frac{\pi w_0^2}{\lambda}\tag{Rayleigh range}$$
$$U_0=\frac{2\alpha}{\varepsilon_0c\pi w_0^2}P\tag{trap depth [Rayleigh approximation]}$$

Near the trap center, a gaussian beam can be well-approximated as harmonic:
$$ U(x,y,z)\approx-U_0+\frac{1}{2}k_xx^2+\frac{1}{2}k_yy^2+\frac{1}{2}k_zz^2\tag{3}$$

The trap stiffnesses $k_x, k_y \text{ and } k_z$ can be obtained from Taylor expanding the full Gaussian potential expression to the second order around the origin. Doing the taylor Taylor expansion we get:
$$ k_x =\frac{4U_0}{w_0^2},\qquad k_y=\frac{4U_0}{w_0^2},\qquad  k_z=\frac{2U_0}{z_R^2}$$

In [8]:
#Rayleigh range, zr 
def rayleigh_range(w0:float, lam: float) -> float: 
    return (np.pi * (w0**2))/lam 

zR = jf.rayleigh_range(w0,wavelength) 

#U0
def trap_depth(alpha: float,P:float,w0:float) -> float:
    return (2*alpha*P)/(8.8541878128e-12*299792458.0*np.pi*(w0**2))
    
U0 = jf.trap_depth(alpha_scalar,laser_power,w0)

def harmonic_stiffness_from_gaussian(
    U0: float,
    w0: float,
    zR: float
) -> tuple[float, float, float]:
    if U0 < 0.0:
        raise ValueError("U0 must be non-negative.")
    if w0 <= 0.0:
        raise ValueError("w0 must be positive.")
    if zR <= 0.0:
        raise ValueError("zR must be positive.")

    kx = 4.0 * U0 / w0**2
    ky = kx
    kz = 2.0 * U0 / zR**2

    return kx, ky, kz

kx, ky, kz = jf.harmonic_stiffness_from_gaussian(U0,w0,zR)

print("Rayleigh range zR =", zR, "m")
print("Trap depth U0 =", U0, "J")

print()
print("Harmonic stiffnesses:")
print("kx =", kx, "N/m")
print("ky =", ky, "N/m")
print("kz =", kz, "N/m")

Rayleigh range zR = 8.657366014854315e-06 m
Trap depth U0 = 1.4091037978393884e-21 J

Harmonic stiffnesses:
kx = 1.5613338480214828e-09 N/m
ky = 1.5613338480214828e-09 N/m
kz = 3.760117558699207e-11 N/m


For a conservative potential, the force is obtained from the negative gradient
of the potential:
$$\mathbf{F}=-\nabla U_{\mathrm{harm}}$$
Then for the harmonic potential
$$\mathbf{F}=(-k_xx,-k_yy,-k_zz)$$
Meaning that the force always points toward the trap center. At the trap center $ \mathbf{R}=(0,0,0)$, the force will be 0; at small displacements it is generated a restoring force proportional to the displacement.

In [10]:
R_test = np.array([
    1.0e-6,
    -1.0e-6,
    1.0e-6
])

def harmonic_com_potential(
    R: np.ndarray,
    kx: float,
    ky: float,
    kz: float
) -> float:
    """
    Potencial armónico tridimensional para el centro de masa.
    """
    R = np.asarray(R, dtype=float)

    if R.shape != (3,):
        raise ValueError("R must be a three-dimensional vector.")

    x, y, z = R

    return 0.5 * (
        kx * x**2
        + ky * y**2
        + kz * z**2
    )

U_test = jf.harmonic_com_potential(R_test,kx,ky,kz)


def harmonic_com_force(
    R: np.ndarray,
    kx: float,
    ky: float,
    kz: float
) -> np.ndarray:
    """
    Fuerza restauradora del potencial armónico.

    F = (-kx*x, -ky*y, -kz*z)
    """
    R = np.asarray(R, dtype=float)

    if R.shape != (3,):
        raise ValueError("R must be a three-dimensional vector.")

    x, y, z = R

    return np.array([
        -kx * x,
        -ky * y,
        -kz * z
    ], dtype=float)




F_test = jf.harmonic_com_force(R_test,kx,ky,kz)

print("R =", R_test, "m")
print("U_harm =", U_test, "J")
print("F_opt =", F_test, "N")



R = [ 1.e-06 -1.e-06  1.e-06] m
U_harm = 1.5801344358149788e-21 J
F_opt = [-1.56133385e-15  1.56133385e-15 -3.76011756e-17] N


At this scale, the viscous effects dominate over the inertia, the traslational motion is modeled in the overdamped regime. For a spherical particle moving through a viscous fluid, the Stokes regime is 
$$\mathbf F_{drag}=-6\pi\eta a_{eff}\mathbf v$$

where $\eta$ is the dynamic viscosity and $a_{eff}$ is an effective particle radius. In this approximation, acceleration is neglected and the optical force is balanced by visscous drag:
$$\mathbf F_{opt}+\mathbf F_{drag}=0$$
Hence
$$\mathbf F_{opt}=6\pi\eta a_{eff}\mathbf v_{opt}$$
$$\therefore \mathbf v_{opt}=\frac{1}{6\pi\eta a_{eff}}\mathbf F_{opt}$$
defining the translational mobility as:
$$\mu_t=\frac{1}{6\pi\eta a_{eff}}$$
We get:
$$\mathbf v_{opt}=\mu_t\mathbf F_{opt}$$

In [12]:
effective_radius = diameter / 2.0

translational_mobility = 1.0 / (
    6.0
    * np.pi
    * eta
    * effective_radius
)

print(
    "Translational mobility =",
    translational_mobility,
    "m/(N s)"
)

v_opt = translational_mobility * F_test
print("Optical drift =", v_opt, "m/s")


Translational mobility = 1590515595.7816954 m/(N s)
Optical drift = [-2.48332584e-06  2.48332584e-06 -5.98052562e-08] m/s


In [15]:
R_test = np.array([
    0.01 * w0,
    -0.01 * w0,
    0.01 * zR
])
F_harmonic = jf.harmonic_com_force(
    R=R_test,
    kx=kx,
    ky=ky,
    kz=kz
)

F_gaussian = jf.gaussian_com_force(
    R=R_test,
    U0=U0,
    w0=w0,
    zR=zR
)

print("Harmonic force =", F_harmonic, "N")
print("Gaussian force =", F_gaussian, "N")

Harmonic force = [-2.96653431e-17  2.96653431e-17 -3.25527140e-18] N
Gaussian force = [-2.96475507e-17  2.96475507e-17 -3.25201778e-18] N


#### Optical orientation
When L is comparable to $w_0$ and since the particle is elongated, different parts of its volume may experience different intensity gradients, rotatiting the particle changes how its volume overlaps with nthe Gaussian field and consequently changes its energy. Independent of any polarizability anisotropy, the potential is proportional to optical intensity of the beam, integrated over the particle volume is:
$$U_{gradient}=-\frac{\alpha_V}{2}\int_V|E(r)|^2dV \tag{4}$$

Where $\alpha_V$ is the polarizability per unit volume. Using the harmonic approximation of eq. (3), Since the beam is radially symetric, this means that the potential changes in the same way wether we move a small distance in the x or y:
$$\therefore\qquad k_x=k_y$$
We define the stiffness of the axis transcerse $k_\perp$ as:
$$k_\perp\equiv k_x=k_y$$
Hence,
$$U(x,y,z)=-U_0+\frac{1}{2}k_\perp(x^2+y^2)+\frac{1}{2}k_zz^2\tag{5}$$
Since the optical energy depends on the local intensity of the electric field (it has not been introduces any anisotropy of polarization yet):
$$U(x,y,z)=-\frac{1}{2}\alpha|E(x,y,z)|^2 \tag{6}$$
Since eq (5) and (6) represent the same potential within our approach we can write:
$$-\frac{1}{2}\alpha|E(x,y,z)|^2=-U_0+\frac{1}{2}k_\perp(x^2+y^2)+\frac{1}{2}k_zz^2$$
Clearing $|E(x,y,z)|^2$ we get:
$$|E(x,y,z)|^2=\frac{2U_0}{\alpha}-\frac{1}{\alpha}[k_\perp(x^2+y^2)+k_zz^2]$$
Substituting into the equation (4) we get:
$$U_{grad}=-\frac{\alpha_v}{2}\int_V\left\{\frac{2U_0}{\alpha}-\frac{1}{\alpha}[k_\perp(x^2+y^2)dV + k_zz^2] \right\}dV$$
$$=-\frac{\alpha_v}{2}\frac{2U_0}{\alpha}\int_VdV+\frac{\alpha_V}{2\alpha}[k_\perp\int_V(x^2+y^2)dV+k_z\int_Vz^2dV$$
Usind $\alpha_V=\alpha/V$ and $\int_VdV=V$ we get:
$$U_{grad}=-U_0+\frac{k_\perp}{2V}\int_V(x^2+y^2)dV+\frac{k_z}{2V}\int_Vz^2dV \tag{7}$$

#### Second spatial moment of volume
To describe how the particle volume is distributed around its center of mass, we introduce the normalized second moment of volume; basically, it measures certain contributions relative to the distance from the center along an axis.
The particle is being modeled as a prolate spheroid with semi-major axis:
$$ a=\frac{L}{2}$$

and transverse semi-axes 
$$b=\frac{D}{2}$$

The elengoted nature of the particle is represented by $a>b$. For a uniform ellipsoid, the normalized second spatial moment along a semi-axis of length is:
$$\frac{1}{V}\int_Vs_\parallel^2dV=\frac{a^2}{5}$$
and the transversal length is:
$$\frac{1}{V}\int_Vs_\perp^2dV=\frac{b^2}{5}$$

In a unitary sphere:
$$\frac{x^2}{b^2}+\frac{y^2}{b^2}+\frac{z^2}{a^2}\leq1$$
Hence,
$$\frac{1}{V}\int_Vs_\parallel^2dV=\frac{1}{V}\int_Vz^2dV=\frac{a^2}{5}$$
$$\frac{1}{V}\int_Vs_\perp^2dV=\frac{1}{V}\int_Vx^2dV=\frac{1}{V}\int_Vy^2dV=\frac{b^2}{5}$$


These quantities represent the mean square distance of the particle volume from its center along each direction.
The long axis of the particle is represented by the unit director $\textbf{p}$. The orientation $\theta$ represents the angle from the z-axis, which corresponds the optical axis; $\theta=0$ means that the particle's long axis lies on the beam axis, whereas $\theta=\pi/2$ means that the particle's long axis lies entirely in the transverse xy-plane.

 Defining the normalized second-moment tensor as:
$$M_{ij}=\frac{1}{V}\int_V\rho_i\rho_j\,dV$$
In 3-D the complete matrix is:
$$
M = \frac{1}{V}
\begin{pmatrix}
\int_Vx^2\text{ }dV & \int_Vxy\text{ }dV & \int_Vxz\text{ }dV\\
\int_Vyx\text{ }dV & \int_Vy^2\text{ }dV & \int_Vyz\text{ }dV\\
\int_Vzx\text{ }dV & \int_Vzy\text{ }dV & \int_Vz^2\text{ }dV
\end{pmatrix} 
$$
Hence
$$
M_{body} = \frac{1}{V}
\begin{pmatrix}
\int_Vx^2\text{ }dV & 0 & 0\\
0 & \int_Vy^2\text{ }dV & 0\\
0 & 0 & \int_Vz^2\text{ }dV
\end{pmatrix}=
\frac{1}{5}
\begin{pmatrix}
b^2 & 0 & 0\\
0 & b^2 & 0\\
0 & 0 & a^2
\end{pmatrix}
$$
It can be expressed as its reduced form in the laboratory frame for a spheroid whose long axis points along $\textbf{p}$:
$$M=\frac{b^2}{5}\mathbf{I} + \frac{a^2-b^2}{5}\mathbf{pp}^T\tag{8}$$
Being:
$$\mathbf{I}=\begin{pmatrix}
1 & 0 & 0\\
0 & 1 & 0\\
0 & 0 & 1
\end{pmatrix},\qquad \mathbf{p}=\begin{pmatrix}
p_x\\
p_y\\
p_z 
\end{pmatrix}\qquad $$
Then the complete Eq. (8) is:
$$M=\frac{b^2}{5}\begin{pmatrix}
1 & 0 & 0\\
0 & 1 & 0\\
0 & 0 & 1
\end{pmatrix} + \frac{a^2-b^2}{5}\begin{pmatrix}
p_x^2 & p_xp_y & p_xp_z\\
p_yp_x & p_y^2 & p_yp_z\\
p_zp_x & p_zp_y & p_z^2
\end{pmatrix}$$
First we want to know the mean-square spread of the particle volume along z, which is:
$$M_{zz}=\frac{1}{V}\int_Vz^2\text{ }dV$$
or
$$M=\frac{b^2}{5}(1)+\frac{a^2-b^2}{5}(p_z^2)$$
Since $p_z=\cos\theta$, then:
$$M_{zz}=\frac{1}{V}\int_Vz^2\text{ }dV=\frac{1}{5}[b^2+(a^2-b^2)\cos^2\theta]\tag{9}$$
Here $\theta$ controls what fraction of the elongation "a" appears in z.

Now, for the transversal part, we want to know $\frac{1}{V}\int_V(x^2+y^2)dV$; we know that for every point in the microalgae, measuring its position with respect to the center is:
$$r=(x,y,z)$$
The square distance from that point to the center is:
$$r^2=x^2+y^2+z^2\qquad \therefore \qquad\frac{1}{V}\int_V(x^2+y^2+z^2)dV = \frac{b^2}{5} +\frac{b^2}{5} + \frac{a^2}{5} $$

Hence

$$\frac{1}{V}\int_V(x^2+y^2+z^2)dV=\frac{a^2+2b^2}{5}\tag{10}$$

To btain $\frac{1}{V}\int_V(x^2+y^2)dV$ we jus have to substract $\frac{1}{V}\int_Vz^2dV$ as
$$\frac{1}{V}\int_V(x^2+y^2)dV=\frac{1}{V}\int_V(x^2+y^2+z^2)dV-\frac{1}{V}\int_Vz^2dV$$

Using Eqs. (7) and (8) we get:
$$\frac{1}{V}\int_V(x^2+y^2)dV=\frac{a^2+2b^2}{5}-\frac{1}{5}[b^2+(a^2-b^2)\cos^2\theta]$$
$$=\frac{1}{5}[a^2+b^2-(a^2-b^2)\cos^2\theta]$$
Using the identity $\sin^2\theta+\cos^2\theta=1\therefore \cos^2\theta= 1 -\sin^2\theta$, then
$$\frac{1}{5}[a^2+b^2-(a^2-b^2)\cos^2\theta]=\frac{1}{5}[a^2+b^2-(a^2-b^2)(1 -\sin^2\theta)]$$
$$=\frac{1}{5}[a^2+b^2-(a^2-\sin^2\theta a^2-b^2+\sin^2\theta b^2)]$$
Hence
$$\frac{1}{V}\int_V(x^2+y^2)dV=\frac{1}{5}[2b^2+(a^2-b^2)\sin^2\theta]\tag{11}$$

###### Summary

Eq. (9) measures the average quadratic spread of the volume along the optical axis z.
$$M_{zz}=\frac{1}{V}\int_Vz^2\text{ }dV=\frac{1}{5}[b^2+(a^2-b^2)\cos^2\theta]\tag{9}$$

Eq. (11) measures the average quadratic spread of the volume in the transverse xy-plane.
$$\frac{1}{V}\int_V(x^2+y^2)dV=\frac{1}{5}[2b^2+(a^2-b^2)\sin^2\theta]\tag{11}$$

#### Connection 
From eq. (7):
$$U_{grad}=-U_0+\frac{k_\perp}{2V}\int_V(x^2+y^2)dV+\frac{k_z}{2V}\int_Vz^2dV \tag{7}$$
Using Eqs. (9) and (11) we get:
$$U_{grad}=-U_0+\frac{k_\perp}{2}\left\{ \frac{1}{5}[2b^2+(a^2-b^2)\sin^2\theta] \right\}+\frac{k_z}{2}\left\{ \frac{1}{5}[b^2+(a^2-b^2)\cos^2\theta] \right\} $$
$$=-U_0+\frac{k_\perp}{10}[2b^2+(a^2-b^2)\sin^2\theta + \frac{k_z}{10}[b^2+(a^2-b^2)\cos^2\theta]$$
$$=-U_0 + \frac{k_\perp2b^2}{10}+\frac{k_\perp(a^2-b^2)}{10}\sin^2\theta+\frac{k_zb^2}{10}+\frac{k_z(a^2-b^2)}{10}\cos^2\theta$$

All terms that do not depend on $\theta$ we will assume that are constants:
$$C=-U_0+\frac{k_\perp2b^2}{10}+\frac{k_zb^2}{10}$$
Hence
$$U_{grad}(0)=C+\frac{a^2-b}{10}[k_\perp\sin^2\theta+k_z\cos^2\theta] \tag{10}$$

if a=b, then $a^2-b^2$ dissappears; on a uniform sphere, there would no be geometric torque.

### Potential depth

Calculating the potential depth:
$$\Delta U_{grad}=U_{grad}(0)-U_{grad}\left(\frac{\pi}{2}\right)$$

Since $\sin^20=0$ and $\cos^20=1$,
$$\therefore \qquad U_{grad}(0)=C+\frac{a^2-b^2}{10}k_z$$

Since $\sin^2\frac{\pi}{2}=1$ and $\cos^2\frac{\pi}{2}=0$,
$$\therefore \qquad U_{grad}(\frac{\pi}{2})=C+\frac{a^2-b^2}{10}k_\perp$$

Hence,
$$\Delta U_{grad}=[C+\frac{a^2-b^2}{10}k_z]-[C+\frac{a^2-b^2}{10}k_\perp]$$
$$\Delta U_{grad}=\frac{a^2-b^2}{10}(k_z-k_\perp)$$

We define the energy cost of rotating the rod from $\theta=0$ to $\theta=\frac{\pi}{2}$ as $\kappa_{grad}$:
$$\kappa_{grad}=\left|\frac{a^2-b^2}{10}(k_z-k_\perp) \right|$$

### Orientation rate

The orientation rate ($\mathbf{p}_{grad}$) that will be added to the orientation rate obtained from the Jeffery's equation ($\mathbf{p}_{Jeff}$) will be obtained through the gradient torque ($\tau_{grad}$), which intrinsically wants to align the director with the z-axis. In a $\textbf{over-damped}$ system (low Reynolds number regime, rotational inertia is disregarded and al torques act at the same time) the gradient torque can be balance by viscous rotational drag,
$$\zeta_r\Omega_{grad}=\tau_{grad}$$
Where $\zeta_r$ is the rotational drag coefficient. Hence
$$\Omega_{grad}=\frac{\tau_{grad}}{\zeta_r}\qquad\therefore\qquad\Omega_{grad}=\frac{2k_{grad}}{\zeta_r}(\mathbf{p}\cdot\hat{\mathbf{z}})(\mathbf{p}\times\hat{\mathbf{z}})$$
Defining 
$$\xi=\frac{2k_{grad}}{\zeta_r}$$
The gradient angular velocity becomes:
$$\Omega_{grad}=\xi(\mathbf{p}\cdot\hat{\mathbf{z}})(\mathbf{p}\times\hat{\mathbf{z}})$$

The change in the orientation vector due to an angular velocity is:
$$\dot{\mathbf{p}}_{grad}=\Omega_{grad}\times\mathbf{p}$$
Hence
$$\dot{\mathbf{p}}_{grad}=\xi(\mathbf{p}\cdot\hat{\mathbf{z}})[(\mathbf{p}\times\hat{\mathbf{z}})\times \mathbf{p}]$$
Using the vector identity
$$(\mathbf{a}\times\hat{\mathbf{b}})\times \mathbf{c}=(\mathbf{a}\cdot\hat{\mathbf{a}})\mathbf{b}-(\mathbf{b}\cdot\hat{\mathbf{c}})\mathbf{a}$$
Then
$$(\mathbf{p}\times\hat{\mathbf{z}})\times \mathbf{p}=\hat{\mathbf{z}}-(\mathbf{p}\cdot\hat{\mathbf{z}})\mathbf{p}$$
Therefore:
$$\dot{\mathbf{p}}_{grad}=\xi(\mathbf{p}\cdot\hat{\mathbf{z}})[\hat{\mathbf{z}}-(\mathbf{p}\cdot\hat{\mathbf{z}})\mathbf{p}]$$
The orientation dynamics ($\dot{\mathbf{p}}$) will be governed by a sum of the contributions described by the jeffery equation and the intensity gradient torque:
$$\dot{\mathbf{p}} = \dot{\mathbf{p}}_{Jeffery} + \dot{\mathbf{p}}_{grad}$$

Let

$$
\boldsymbol{\rho}=
\begin{pmatrix}
\rho_x\\
\rho_y\\
\rho_z
\end{pmatrix}
$$

denote the position of a point inside the microalga measured relative to its
center of mass.